# nn.Module and nn.Sequential

## you can define your own layer more freely!(by using nn.Module)


In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F


In [2]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden=nn.Linear(20,256)  #insrease the dimision to find the inner connection
        self.out=nn.Linear(256,10)
    def forward(self,x):
        y_pred=self.out(F.relu(self.hidden(x)))  #simplified use 
#use self.out and self.hidden function to forward calculate
#init function is to define the instruction,while forward function use it (or not)
#BUT FORWARD CAN ALSO HAVE SOME SELF_DEFINE USE OF FUNCTIONS:
    def forward(self,x):
        x=self.hidden(x)
        x=F.relu(x)# this is not defined in __init__ function!!
        x=self.out(x)
        return x
#remember "__init__"  hard code ：in python,constructor is required to write as"__init__"
#use relu visibly instead of define it in __init__()
#self is used to act as a insitantiate object ,x is the key:input 

In [3]:
net=MLP()
x=torch.rand(2,20)
y=net(x)


Layer:nn.sequential  
Block:nn.Module  
seauential is always used in Module class's member function:' __ init __ ' to simplify the express:
```python
self.fc=nn.Sequential(nn.Linear(20,256),nn.relu(),nn.Linear(256,10))
```
in FORWARD function ,just use: 
```python
return self.fc(x)
```
the function will automatically realize forward layer by layer

the implement of Sequential function is as follows:

In [4]:
class MySequential(nn.Module):
    def __init__(self,*args):   #*args:list of input arguments
        #the *args will receive all the parameters and pack it to a tuple(元组)
        #e.g.:Myclass(a,b,c),then args is (a,b,c)
        super().__init__()
        #as (a,b,c) is args,the for loop is to traverse every element in args
        #and save it as both key and value in dictionary self.modules
        for block in args:
            self._modules[block]=block

    def forward(self,x):
            for block in self._modules.values():
                x=block(x)
            return x

net=MySequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
net(x)
#python is a dynamic language so no matter what kind the real parameter of args is
#it can always work

tensor([[ 0.3061,  0.1562,  0.1019, -0.0055, -0.1657,  0.2052,  0.0343, -0.0601,
          0.0416,  0.1432],
        [ 0.3110,  0.3180, -0.0031, -0.0558, -0.0359,  0.1885, -0.0293, -0.0502,
         -0.0338,  0.0150]], grad_fn=<AddmmBackward0>)

## module and sequential can also be mixed

In [5]:
net=nn.Sequential(MLP(),nn.Linear(10,20),nn.ReLU(),MLP())
net(x)
#nn.Sequential can receive any son-class of nn.Module

tensor([[ 0.0275, -0.0185,  0.0215, -0.0827,  0.0041,  0.0020, -0.0155, -0.0043,
         -0.0283,  0.0294],
        [ 0.0232, -0.0138,  0.0241, -0.0814,  0.0021,  0.0052, -0.0121, -0.0035,
         -0.0241,  0.0301]], grad_fn=<AddmmBackward0>)

In [9]:
#mixed use:
layer=nn.Sequential(MLP(),nn.Linear(10,256),nn.ReLU(),nn.Linear(256,10))
class Mixed(nn.Module):   #DONT FORGET TO INHERIT nn.Module!!!!
    def __init__(self):
        super().__init__()
        self.fc=nn.Sequential(     #squential used in module(module_sequential)
            layer,      #sequential used in module_sequential
            nn.Linear(10,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )
    def forward(self,x):
        return self.fc(x)

net=Mixed()
net(x)

tensor([[ 0.0845, -0.0234, -0.0084,  0.1495, -0.0630,  0.0810,  0.1634,  0.0667,
         -0.0007, -0.0125],
        [ 0.0837, -0.0246, -0.0082,  0.1501, -0.0632,  0.0726,  0.1599,  0.0693,
          0.0058, -0.0135]], grad_fn=<AddmmBackward0>)